In [17]:
import os
import boto3
from sagemaker import get_execution_role
import shutil
from pprint import pprint
import time

### Constants

In [18]:
# function name
str_function_name = 'christian-concat-tuning'

### 1. Create container

### Create ```Dockerfile```

In [19]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [20]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0
pandas==1.2.4
boto3==1.24.59

Writing requirements.txt


### Write ```lambda_function.py```

In [21]:
%%writefile lambda_function.py

import pandas as pd
import numpy as np
import boto3

# lambda handler
def lambda_handler(event, context):
    # constants
    str_project = '20240509-christian-internship'
    str_model = '09_aws_batch'
    str_prefix = f'{str_model}/output'

    # # get df_hyperparameters
    # str_filename = 'df_hyperparameters.csv'
    # str_uri = f's3://{str_project}/{str_model}/02_model/01_feat_select/05_step_function/{str_filename}'
    # df = pd.read_csv(str_uri)
    # # convert to dict
    # dict_hyperparameters = dict(zip(df['keys'], df['values']))
    
    # # get eval metric
    # str_eval_metric = dict_hyperparameters['STR_EVAL_METRIC']

    # hard code eval metric
    str_eval_metric = 'AUC'
    print(f'Eval metric: {str_eval_metric}')
    
    # get index files in s3
    print('Getting files in s3...')
    cls_client = boto3.resource('s3')
    cls_bucket = cls_client.Bucket(str_project)
    list_str_filenames = []
    for file in cls_bucket.objects.filter(Prefix=str_prefix):
        # get key
        str_key = file.key
        # make sure it is a .csv
        if '.csv' in str_key:
            # get filename
            str_filename = str_key.split('/')[-1]
            list_str_filenames.append(str_filename)
    print(f'There are {len(list_str_filenames)} files to import')
    
    # iterate and import
    print('Importing files...')
    list_df = []
    for str_filename in list_str_filenames:
        str_uri = f's3://{str_project}/{str_prefix}/{str_filename}'
        df = pd.read_csv(str_uri)
        list_df.append(df)
    
    # create df
    print('Creating data frame...')
    df = pd.concat(list_df)
    del list_df
    
    # logic for sorting
    print('Sorting...')
    if str_eval_metric in ['AUC', 'PRAUC', 'F1']:
        bool_ascending = False
    else:
        bool_ascending = True
        
    # sort
    df.sort_values(by='flt_eval_metric_valid', ascending=bool_ascending, inplace=True)
    
    # write to s3
    print('Writing to s3...')
    str_filename = 'df_tuning.csv'
    str_uri = f's3://{str_project}/08_aws_lambda/output/{str_filename}'
    df.to_csv(str_uri, index=False)

Writing lambda_function.py


### Build image and push to ECR

In [22]:
%%sh

# name the image
image=christian-concat-tuning

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  66.05kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
 ---> f0da30445c8b
Step 2/6 : RUN pip install --upgrade pip
 ---> Using cache
 ---> e717ea60aa7b
Step 3/6 : COPY requirements.txt  .
 ---> Using cache
 ---> a548c643b808
Step 4/6 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Using cache
 ---> cd05e49cf9b7
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> Using cache
 ---> 223db0c6cb47
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Using cache
 ---> 1f1d7be60d56
Successfully built 1f1d7be60d56
Successfully tagged christian-concat-tuning:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'christian-concat-tuning' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/christian-concat-tuning]
60294554a2e2: Preparing
6a0f900d56ac: Preparing
2ae2acb5092c: Preparing
a3fef5c8f203: Preparing
1d4a83077be2: Preparing
d3f396c66075: Preparing
fbbd8c1e2ec1: Preparing
15b9c146ec8f: Preparing
d716890aa5e0: Preparing
ecafc03674ff: Preparing
d3f396c66075: Waiting
fbbd8c1e2ec1: Waiting
15b9c146ec8f: Waiting
d716890aa5e0: Waiting
ecafc03674ff: Waiting
6a0f900d56ac: Layer already exists
2ae2acb5092c: Layer already exists
60294554a2e2: Layer already exists
a3fef5c8f203: Layer already exists
1d4a83077be2: Layer already exists
d3f396c66075: Layer already exists
fbbd8c1e2ec1: Layer already exists
ecafc03674ff: Layer already exists
d716890aa5e0: Layer already exists
15b9c146ec8f: Layer already exists
latest: digest: sha256:b6c5a59fad9a8c1b717a512da756786f2e73e877f58e77f7c464cfea6ddceeac size: 2420


### 2. Create lambda function from image

In [23]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [24]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [26]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Wed, 12 Jun 2024 19:17:56 GMT',
                                      'x-amzn-requestid': '0566f160-b536-4353-b530-58d2a937cffd'},
                      'HTTPStatusCode': 204,
                      'RequestId': '0566f160-b536-4353-b530-58d2a937cffd',
                      'RetryAttempts': 0}}


In [27]:
# create function
str_image_uri = '836690756591.dkr.ecr.us-west-2.amazonaws.com/christian-concat-tuning:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=60,
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': 'b6c5a59fad9a8c1b717a512da756786f2e73e877f58e77f7c464cfea6ddceeac',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:christian-concat-tuning',
 'FunctionName': 'christian-concat-tuning',
 'LastModified': '2024-06-12T19:17:58.698+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/christian-concat-tuning'},
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1201',
                                      'content-type': 'application/json',
                                      'date': 'Wed, 12 Jun 2024 19:17:59 GMT',
                                      'x-amzn-requestid': 'fe55985d-79a0-447a-974e-1af5179b6fd2'},
                      'HTTPStatusCode': 201,
                      'RequestId': 'fe55985d-79a0

### Clean-up

In [28]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)